In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

# 1. Load the dataset and fix header encoding artifacts
try:
    df = pd.read_csv("/content/Dataset.csv", encoding="latin1")
    # Strip hidden characters like ï»¿ from headers
    df.columns = df.columns.str.replace(r'[^\w\s\(\)]', '', regex=True).str.strip()
    print("Dataset successfully loaded and column headers standardized!")
except Exception as e:
    print(f"Error loading file: {e}. Please ensure 'Dataset.csv' is uploaded.")

# 2. Complete Feature Engineering Pipeline matching the Problem Statement Requirements
def engineer_customer_features(df):
    customer_df = df.copy()

    # Feature 1: Promo Dependency Score (1.0 if discount used, else 0.0)
    customer_df['dependency_score'] = customer_df['Discount Applied'].apply(lambda x: 1.0 if x == 'Yes' else 0.0)

    # Feature 2: Customer Lifetime Value (CLV) Proxy
    # Current spend multiplied by total transaction count history
    customer_df['clv_proxy'] = customer_df['Purchase Amount (USD)'] * (customer_df['Previous Purchases'] + 1)

    # Categorize Value Tiers using clear statistical quantiles (33rd and 66th percentiles)
    q33, q66 = customer_df['clv_proxy'].quantile([0.33, 0.66])

    def assign_value_tier(spend):
        if spend <= q33:
            return 'Low-Value'
        elif spend <= q66:
            return 'Mid-Value'
        else:
            return 'High-Value'

    customer_df['value_tier'] = customer_df['clv_proxy'].apply(assign_value_tier)

    # Impute missing review values with the dataset median to avoid data distortion
    customer_df['Review Rating'] = customer_df['Review Rating'].fillna(customer_df['Review Rating'].median())

    # Feature 3: Satisfaction Flag (Below 3.5 denotes operational risk)[cite: 1]
    customer_df['satisfaction_flag'] = np.where(customer_df['Review Rating'] < 3.5, 'Low-Satisfaction', 'Satisfied')

    # Central Challenge: Competing Loyalty Definitions[cite: 1]
    # Definition A: Volume/Frequency-Driven (High historical baseline)[cite: 1]
    customer_df['loyalty_definition_a'] = np.where(customer_df['Previous Purchases'] >= 25, 'Loyal', 'Casual')

    # Definition B: Margin/Organic-Driven (High Lifetime Spend + Zero Discount Reliance)[cite: 1]
    customer_df['loyalty_definition_b'] = np.where(
        (customer_df['clv_proxy'] >= customer_df['clv_proxy'].median()) & (customer_df['Discount Applied'] == 'No'),
        'Loyal',
        'Casual'
    )

    return customer_df

# 3. Execute data modification and trigger download
try:
    enriched_df = engineer_customer_features(df)

    # Save optimized dataset containing Category and Location metrics for Power BI[cite: 1]
    output_filename = '/content/enriched_customer_data.csv'
    enriched_df.to_csv(output_filename, index=False)

    print("\n--- Python Data Optimization Success! ---")
    print(f"File created with dimensional variables intact: {output_filename}")
    print("Columns ready for SQL/Power BI:", list(enriched_df.columns))

    # Trigger download
    files.download(output_filename)
except Exception as e:
    print(f"\nAn error occurred: {e}")

Dataset successfully loaded and column headers standardized!

--- Python Data Optimization Success! ---
File created with dimensional variables intact: /content/enriched_customer_data.csv
Columns ready for SQL/Power BI: ['ïCustomer ID', 'Age', 'Gender', 'Item Purchased', 'Category', 'Purchase Amount (USD)', 'Location', 'Size', 'Color', 'Season', 'Review Rating', 'Subscription Status', 'Shipping Type', 'Discount Applied', 'Promo Code Used', 'Previous Purchases', 'Payment Method', 'Frequency of Purchases', 'dependency_score', 'clv_proxy', 'value_tier', 'satisfaction_flag', 'loyalty_definition_a', 'loyalty_definition_b']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>